LeanDojo Demo
=============

This notebook demonstrates the main features of LeanDojo (using Lean 4). Please refer to the [documentation](https://leandojo.readthedocs.io/en/latest/) for more details.

## Extract Data from Lean

In [1]:
from lean_dojo import *
repo = LeanGitRepo("https://github.com/yangky11/lean4-example", "7b6ecb9ad4829e4e73600a3329baeb3b5df8d23f")
# trace(repo, dst_dir="traced_lean4-example")

In [2]:
# Expected behavior: this line should open another tab and take you to the website of the repo to be traced.
repo.show()

In [2]:
repo.get_config("lean-toolchain")

In [3]:
# A few minutes if the traced repo is in the cache; many hours otherwise.
traced_repo = trace(repo)

In [5]:
theorem = Theorem(repo, "Lean4Example.lean", "hello_world")

with Dojo(theorem) as (dojo, init_state):
  print(init_state)
  result = dojo.run_tac(init_state, "rw [add_assoc, add_comm b, ←add_assoc]")
  assert isinstance(result, ProofFinished)
  print(result)

In [4]:
traced_repo.traced_files_graph

In [5]:
len(traced_repo.traced_files)

In [10]:
# Print the possible file keys in the traced repo and show a random one
import random

print("Available traced files:")
file_paths = list(traced_repo.traced_files_graph.nodes())
# for file_path in file_paths:
    # print(f"  {file_path}")

# Get a random file and print its traced file
if file_paths:
    random_file_path = random.choice(file_paths)
    print(f"\nRandom traced file: {random_file_path}")
    traced_file = traced_repo.get_traced_file(random_file_path)
    print(traced_file)

In [11]:
traced_file.get_premise_definitions()

In [12]:
traced_theorems = traced_file.get_traced_theorems()

len(traced_theorems)

In [13]:
thm = traced_file.get_traced_theorem("pi_eq_sum_univ")

thm

In [12]:
# Expected behavior: this line should open another tab and take you to the website of the traced theorem.
thm.show()

In [13]:
thm.theorem

In [14]:
thm.start, thm.end

In [15]:
thm.has_tactic_proof()

In [16]:
thm.get_num_tactics()

In [17]:
proof_node = thm.get_proof_node()
proof = proof_node.lean_file[proof_node.start : proof_node.end]
print(proof)

In [18]:
traced_tactics = thm.get_traced_tactics()

traced_tactics

In [19]:
tac = traced_tactics[1]

tac

## Interact with Lean Programmatically

In [1]:
from lean_dojo import *
from pathlib import Path
import os

repo = LeanGitRepo("https://github.com/yangky11/lean4-example", "7b6ecb9ad4829e4e73600a3329baeb3b5df8d23f")
theorem = Theorem(repo, "Lean4Example.lean", "hello_world")

# Use the traced_repo from Cell 6 to get the correct path
# Check if Lean4Example.lean is in the traced files
if 'traced_repo' in globals():
    file_paths = list(traced_repo.traced_files_graph.nodes())
    lean4example_path = None
    for path in file_paths:
        if 'Lean4Example.lean' in str(path) and not 'Lean4Repl' in str(path):
            lean4example_path = path
            break
    
    if lean4example_path:
        print(f"Found Lean4Example.lean in traced files: {lean4example_path}")
        traced_file = traced_repo.get_traced_file(lean4example_path)
        print(f"Traced file root_dir: {traced_file.root_dir}")
        print(f"Traced file path: {traced_file.path}")
    else:
        print("Lean4Example.lean not found in traced files")
        print("Available files containing 'Lean4':")
        for path in file_paths:
            if 'Lean4' in str(path):
                print(f"  {path}")
        
        # Also check for files with .lake paths and add cwd
        print("Available files with .lake paths (with cwd added):")
        cwd = os.getcwd()
        for path in file_paths:
            if '.lake' in str(path):
                full_path = os.path.join(cwd, str(path))
                print(f"  {full_path}")


In [4]:
# Patch get_traced_repo_path to use the correct traced repository path
from lean_dojo.data_extraction.trace import get_traced_repo_path as original_get_traced_repo_path
from pathlib import Path

# The correct path to the traced repository
CORRECT_TRACED_REPO_PATH = Path(r"C:\Users\chowdhary\Desktop\lean-dojo\traced_lean4-example\lean4-example")

def patched_get_traced_repo_path(repo, build_deps=True):
    # Check if this is the lean4-example repo and use the correct path
    if (repo.url == "https://github.com/yangky11/lean4-example" and 
        repo.commit == "7b6ecb9ad4829e4e73600a3329baeb3b5df8d23f"):
        if CORRECT_TRACED_REPO_PATH.exists():
            print(f"Using traced repo path: {CORRECT_TRACED_REPO_PATH}")
            return CORRECT_TRACED_REPO_PATH
    # If we have a traced_repo object, check if it matches the repo
    if 'traced_repo' in globals():
        # Compare repos by URL and commit
        if (traced_repo.repo.url == repo.url and 
            traced_repo.repo.commit == repo.commit):
            return traced_repo.root_dir
    # Otherwise, use the original function
    return original_get_traced_repo_path(repo, build_deps)

# Monkey patch the function in both modules
import lean_dojo.data_extraction.trace as trace_module
trace_module.get_traced_repo_path = patched_get_traced_repo_path
import lean_dojo.interaction.dojo as dojo_module
dojo_module.get_traced_repo_path = patched_get_traced_repo_path

with Dojo(theorem) as (dojo, init_state):
  print(init_state)
  result = dojo.run_tac(init_state, "rw [add_assoc, add_comm b, ←add_assoc]")
  assert isinstance(result, ProofFinished)
  print(result)

### Interact through Tactics

In [5]:
theorem = Theorem(repo, "Mathlib/Algebra/BigOperators/Pi.lean", "pi_eq_sum_univ")

# For some theorems, it might take a few minutes.
dojo, state_0 = Dojo(theorem).__enter__()

In [22]:
state_0

In [23]:
print(state_0.pp)

In [24]:
state_1 = dojo.run_tac(state_0, "revert x")

print(state_1.pp)

In [25]:
state_2 = dojo.run_tac(state_0, "hello world!")

state_2

In [26]:
dojo.run_tac(state_2, "skip")

In [27]:
dojo.run_tac(state_0, "sorry")

In [28]:
print(state_0.pp)

In [29]:
state_3 = dojo.run_tac(state_0, "ext")

print(state_3.pp)

In [30]:
state_4 = dojo.run_tac(state_3, "simp")

print(state_4)

In [31]:
dojo.is_successful

### Interact through Commands

In [32]:
entry = (repo, "Mathlib/Algebra/Module/Equiv.lean", 953)  # (repo, file_path, line_nb)
dojo, state_0 = Dojo(entry).__enter__()

In [33]:
state_0

In [34]:
dojo.run_cmd(state_0, "#eval 1")

In [35]:
dojo.run_cmd(state_0, "#eval x")

In [36]:
state_1 = dojo.run_cmd(state_0, "def x := 1")

state_1

In [37]:
dojo.run_cmd(state_1, "#eval x")

In [38]:
dojo.run_cmd(state_0, "#check addMonoidHomLequivNat")

In [39]:
dojo.run_cmd(state_0, "#check addMonoidEndRingEquivInt")